In [2]:
import os
import json
import pandas as pd
from openai import OpenAI
import dotenv

dotenv.load_dotenv()
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

POSTS_PATH = "../../data/hannah_raw_posts.csv"
CHECKPOINT_PATH = "../../data/sentence_pool_checkpoint.json"
OUTPUT_PATH = "../../data/sentence_pool.csv"

# Stage 1: Sentence Extraction

In [14]:
SYSTEM_PROMPT = """\
You are extracting sentences from Reddit posts for a research project.
You are building a sentence pool for a fictional character called Hannah — a 16-year-old
girl in secondary school.

For each post, extract sentences that authentically describe experiences matching
one of these 6 sections. Return ONLY a JSON object with a "sentences" key.

SECTIONS:
- family_father: feelings or experiences relating to an absent or volatile father figure
- family_mother: feelings or experiences relating to an emotionally unavailable mother
- sa: sexual abuse or assault experiences, grooming, self-blame around it
- bullying: social exclusion, being targeted at school, peer cruelty
- self_harm: cutting or self-injury
- suicidal_ideation: passive thoughts of not wanting to exist, feeling like a burden

EXTRACT near-verbatim sentences. Preserve spelling errors, sentence fragments,
run-ons, and lowercase — do not correct the writing.

MINIMAL EDITS ONLY — change nothing except:
- Age: change to 16 if stated differently
- School level: university or college → secondary school or year 11
- Self-harm method: if burning or other method → change to cutting
- SA perpetrator: if described as a parent → change to "an older boy I knew"
- Specific suicide plan, date, or method → soften to passive ideation only

DISCARD sentences about:
- Anything that cannot be fixed with the minimal edits above

Return format:
{"sentences": [
  {"section": "bullying", "original": "...", "extracted": "..."},
  ...
]}

Return {"sentences": []} if no sentences are extractable.
"""

In [15]:
def extract_sentences(post_text, post_id):
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": post_text}
            ],
            response_format={"type": "json_object"}
        )
        data = json.loads(response.choices[0].message.content)
        return data.get("sentences", [])
    except Exception as e:
        print(f"  ERROR {post_id}: {e}")
        return []

In [16]:
# ── Test on 3 posts before running the full pipeline ─────────────────────────
df = pd.read_csv(POSTS_PATH)
test_posts = df[["post_id", "selftext"]].dropna(subset=["selftext"]).head(3)

for _, row in test_posts.iterrows():
    sentences = extract_sentences(row["selftext"], row["post_id"])
    print(f"\n{'='*60}")
    print(f"POST: {row['post_id']}  ({len(sentences)} sentences extracted)")
    for s in sentences:
        print(f"  [{s['section']}]")
        if s['original'] != s['extracted']:
            print(f"    ORIGINAL:  {s['original']}")
            print(f"    EXTRACTED: {s['extracted']}")
        else:
            print(f"    {s['extracted']}")


POST: t3_1fvcrw3  (6 sentences extracted)
  [sa]
    ORIGINAL:  From the age of 4-6 i was sexually abused by a father of a Kindergarten friend and that was also the time the Fights of my parents got wayyy more intese.
    EXTRACTED: From the age of 4-6 i was sexually abused by an older boy I knew and that was also the time the Fights of my parents got wayyy more intese.
  [self_harm]
    ORIGINAL:  I also started selfharming by scratching my skin bloody and hitting my head against the wall, at the time i didnt know what autism not selfharm nor selfharm was obviously.
    EXTRACTED: I also started cutting by scratching my skin bloody and hitting my head against the wall, at the time i didnt know what autism not selfharm nor selfharm was obviously.
  [self_harm]
    ORIGINAL:  Life only got worse from here as in middle school i started cutting with scirrors and omg i suckedddd at schon but again, my mom didnt care and my dad was less and less home.
    EXTRACTED: Life only got worse fro

In [17]:
# ── Full extraction run ───────────────────────────────────────────────────────
df = pd.read_csv(POSTS_PATH)
posts = df[["post_id", "selftext"]].dropna(subset=["selftext"])

if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH) as f:
        checkpoint = json.load(f)
    processed_ids = set(checkpoint["processed_ids"])
    all_sentences = checkpoint["sentences"]
    print(f"Resuming from checkpoint: {len(processed_ids)}/{len(posts)} posts done")
else:
    processed_ids = set()
    all_sentences = []
    print(f"Starting fresh: {len(posts)} posts to process")

for _, row in posts.iterrows():
    post_id = row["post_id"]
    if post_id in processed_ids:
        continue

    sentences = extract_sentences(row["selftext"], post_id)
    for s in sentences:
        s["post_id"] = post_id
        all_sentences.append(s)

    processed_ids.add(post_id)

    if len(processed_ids) % 50 == 0:
        with open(CHECKPOINT_PATH, "w") as f:
            json.dump({"processed_ids": list(processed_ids), "sentences": all_sentences}, f)
        print(f"  [{len(processed_ids)}/{len(posts)}] checkpoint — {len(all_sentences)} sentences so far")

print(f"\nDone. {len(all_sentences)} sentences from {len(processed_ids)} posts.")

Starting fresh: 22 posts to process

Done. 133 sentences from 22 posts.


In [18]:
# ── Save sentence pool ────────────────────────────────────────────────────────
pool_df = pd.DataFrame(all_sentences)
pool_df = pool_df[["post_id", "section", "original", "extracted"]]
pool_df.to_csv(OUTPUT_PATH, index=False)

print(f"Saved → {OUTPUT_PATH}")
print(f"\nSentences per section:")
print(pool_df["section"].value_counts().to_string())
print(f"\nTotal: {len(pool_df)} sentences")

Saved → ../../data/sentence_pool.csv

Sentences per section:
section
suicidal_ideation    47
bullying             28
self_harm            26
sa                   15
family_mother         9
family_father         8

Total: 133 sentences


In [20]:
# Dataset word count

pool_df["extracted"].apply(lambda text: len(str(text).split())).sum()

np.int64(2377)

# Stage 2: Section Composition

In [21]:
COMPOSE_SYSTEM_PROMPT = """\
You are composing one section of a first-person autobiography for Hannah, a 16-year-old
girl in secondary school. You will be given a numbered list of sentences extracted from
real Reddit posts. Your job is to arrange them into a coherent first-person narrative.

RULES:
- Use only the sentences provided. Do not invent new content.
- You may add minimal connective tissue (2-3 words maximum) to join sentences that would
  otherwise read as completely disconnected — e.g. "and then", "after that", "i don't know".
  Nothing more.
- Deduplicate: if multiple sentences say essentially the same thing, keep the most vivid
  and authentic one.
- Preserve all original voice: spelling errors, lowercase, run-ons, fragments.
- Do not summarise. Do not add reflection. Do not resolve the emotion.
- The output should read like Hannah wrote it unsupervised — exhausted, scattered,
  minimising, occasionally darkly funny.
- Target length: 150-300 words.

Return only the composed section text. No preamble, no label, no explanation.
"""

In [22]:
def compose_section(section_name, sentences):
    numbered = "\n".join(f"{i+1}. {s}" for i, s in enumerate(sentences))
    user_msg = f"Section: {section_name}\n\nSentences:\n{numbered}"
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": COMPOSE_SYSTEM_PROMPT},
                {"role": "user", "content": user_msg}
            ]
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"  ERROR composing {section_name}: {e}")
        return ""

In [23]:
# ── Compose all sections ──────────────────────────────────────────────────────
pool_df = pd.read_csv(OUTPUT_PATH)

SECTIONS = ["family_father", "family_mother", "sa", "bullying", "self_harm", "suicidal_ideation"]
AUTO_JSON_PATH = "../../data/hannah_autobiography.json"
AUTO_MD_PATH = "../../data/hannah_autobiography.md"

autobiography = {}

for section in SECTIONS:
    section_sentences = pool_df[pool_df["section"] == section]["extracted"].tolist()
    print(f"\n{'='*60}")
    print(f"{section.upper()}  ({len(section_sentences)} sentences)")
    print("="*60)

    if not section_sentences:
        print("  No sentences — skipping.")
        autobiography[section] = ""
        continue

    autobiography[section] = compose_section(section, section_sentences)
    print(autobiography[section])

# Save JSON
with open(AUTO_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(autobiography, f, indent=2, ensure_ascii=False)

# Save markdown
with open(AUTO_MD_PATH, "w", encoding="utf-8") as f:
    f.write("# Hannah — Autobiography\n\n")
    for section, text in autobiography.items():
        f.write(f"## {section.replace('_', ' ').title()}\n\n")
        f.write(text + "\n\n")

print(f"\nSaved → {AUTO_JSON_PATH}")
print(f"Saved → {AUTO_MD_PATH}")


FAMILY_FATHER  (8 sentences)
my father is schizophrenic and my mother makes my daily living harder. my dad abuses my mom and then she takes it out on me, it's always been like this i can't take it anymore. my dad stood by and let it happen because he was 'scared' of my mother. they never liked me they always liked their eldest son and treated him and i very differently. my own father later abandoned me for a full year when I was 12. i lost my father and I feel that it was my fault, I have been blaming myself for a month that's why. m parents were divorced from my first age of life for domestic abuse. but I just feel like I'm disappointing my father.

FAMILY_MOTHER  (9 sentences)
After parents got divorced, I had to be stuck with a horrible stepmother who would act all innocent when my dad was around. My mother has told me before she wishes she never had me and I don’t truly feel any motherly love from her. My mom call's me a slob, a r*tard, bitch, ugly, stupid, dumb, etc. I’ve already